# Training Setup

This notebook prepares the full training pipeline for TwinCar's car make/model/year
classifier. It operates exclusively on Stanford Cars (16,103 images, 195 classes)
using the split defined in `stanford_metadata.csv`.

Steps covered:
1. Copy images from HuggingFace cache into ImageFolder-compatible folder structure
2. Verify split integrity
3. Define augmentation strategy
4. Build DataLoaders with class-balance sampling
5. Load and configure EfficientNet-B0 for fine-tuning
6. Define loss, optimizer, and LR scheduler
7. Sanity-check a single forward pass

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import shutil
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
import json

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B0_Weights

from datasets import load_dataset

PROJECT_DIR = Path("/content/drive/MyDrive/twincar")  # Drive project folder
DATA_DIR = Path("/content/twincar_data")              # local Colab storage for copied images

METADATA_PATH = PROJECT_DIR / "data" / "stanford_metadata.csv"
STANFORD_CACHE = PROJECT_DIR / "stanford_cars_cache"

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("METADATA_PATH exists:", METADATA_PATH.exists())
print("STANFORD_CACHE:", STANFORD_CACHE)
print("MODEL_DIR:", MODEL_DIR)

assert PROJECT_DIR.exists(), "PROJECT_DIR does not exist. Check Google Drive mount."
assert METADATA_PATH.exists(), "stanford_metadata.csv not found. Run 02_data_preparation first."

PROJECT_DIR: /content/drive/MyDrive/twincar
DATA_DIR: /content/twincar_data
METADATA_PATH exists: True
STANFORD_CACHE: /content/drive/MyDrive/twincar/stanford_cars_cache
MODEL_DIR: /content/drive/MyDrive/twincar/models


## Folder Structure Strategy

We copy images from the HuggingFace cache into a directory tree that matches
PyTorch's `ImageFolder` convention:

    data/
      train/
        AM_General_Hummer_SUV_2000/
        Acura_Integra_Type_R_2001/
        ...
      val/
        ...
      test/
        ...

`ImageFolder` auto-assigns labels from subfolder names, which means:
- No custom Dataset class needed
- Fully compatible with torchvision pipelines
- Easy to inspect on disk during debugging

Class names are sanitized (spaces -> underscores, slashes removed) to be
filesystem-safe. The label integer from `stanford_metadata.csv` is NOT used
directly - `ImageFolder` will re-assign 0-indexed integers alphabetically,
which is consistent and reproducible.

In [3]:
metadata = pd.read_csv(METADATA_PATH)

def sanitize(name):
    return name.replace(" ", "_").replace("/", "-")

def reset_split_dirs():
    """
    Remove old ImageFolder split directories so previous runs
    do not leave stale images inside train/val/test.
    """
    for split_name in ["train", "val", "test"]:
        split_dir = Path(DATA_DIR) / split_name

        if split_dir.exists():
            shutil.rmtree(split_dir)

        split_dir.mkdir(parents=True, exist_ok=True)

    print("Old train/val/test folders cleared.")

def get_class_name(row):
    """
    Use class_name if available, otherwise fallback to car_name.
    """
    if "class_name" in row and pd.notna(row["class_name"]):
        return row["class_name"]
    return row["car_name"]

def copy_split(dataset_hf, df_split, split_name):
    split_dir = Path(DATA_DIR) / split_name
    copied = 0

    for _, row in df_split.iterrows():
        class_name = get_class_name(row)
        class_dir = split_dir / sanitize(class_name)
        class_dir.mkdir(parents=True, exist_ok=True)

        out_path = class_dir / f"{int(row['hf_idx'])}.jpg"

        img = dataset_hf[int(row["hf_idx"])]["image_path"]
        img.save(out_path)

        copied += 1

    print(f"{split_name}: {copied} images copied")

dataset = load_dataset("naufalso/stanford_cars", cache_dir=STANFORD_CACHE)

train_df = metadata[metadata["split"] == "train"].reset_index(drop=True)
val_df   = metadata[metadata["split"] == "val"].reset_index(drop=True)
test_df  = metadata[metadata["split"] == "test"].reset_index(drop=True)

reset_split_dirs()

copy_split(dataset["train"], train_df, "train")
copy_split(dataset["train"], val_df,   "val")
copy_split(dataset["test"],  test_df,  "test")

del dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/614 [00:00<?, ?B/s]

Old train/val/test folders cleared.
train: 6887 images copied
val: 1216 images copied
test: 8000 images copied


In [5]:
EXPECTED_CLASSES = 195

expected_counts = {
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df)
}

for split in ["train", "val", "test"]:
    split_path = Path(DATA_DIR) / split
    classes = sorted(os.listdir(split_path))

    total_imgs = sum(
        len(list((split_path / c).glob("*.jpg")))
        for c in classes
    )

    print(f"{split}: {len(classes)} classes, {total_imgs} images")

    assert len(classes) == EXPECTED_CLASSES, f"{split} should have {EXPECTED_CLASSES} classes"
    assert total_imgs == expected_counts[split], (
        f"{split} should have {expected_counts[split]} images, but found {total_imgs}"
    )

print("Folder verification passed.")

train: 195 classes, 6887 images
val: 195 classes, 1216 images
test: 195 classes, 8000 images
Folder verification passed.


## Augmentation Strategy

Stanford Cars contains relatively clean public car images, while the real TwinCar use case may include more variation from drones and ground robots.

The production images may contain:

- different camera angles
- lighting changes
- shadows and reflections
- different vehicle scale in the image
- small camera tilt

For the first baseline, we use conservative augmentation. The goal is to make the model more robust without destroying important make/model details such as headlights, grille shape, rear lights, and body shape.

| Transform | Purpose |
|---|---|
| `RandomResizedCrop(224, scale=0.8–1.0)` | Simulates small scale and crop variation |
| `RandomHorizontalFlip(p=0.5)` | Handles left/right vehicle direction |
| `RandomRotation(10°)` | Simulates small camera tilt |
| `ColorJitter(0.2, 0.2, 0.1, 0.02)` | Simulates moderate lighting and color variation |

Validation and test images use deterministic preprocessing only. No random augmentation is applied to validation or test data, because we want stable and fair evaluation metrics.

In [6]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## Handling Class Imbalance

The EDA found class counts ranging from 24 to 68 images - a ~2.8× ratio.
While not extreme, with only ~35 images/class on average, the majority classes
can still dominate gradient updates during early training.

We use `WeightedRandomSampler` rather than weighted loss for the following reason:
sampler-based oversampling ensures every batch sees a balanced class distribution,
which stabilizes gradient variance early in training. Weighted loss is complementary
but acts more softly — it scales gradients rather than guaranteeing exposure.

For a mild imbalance like ours, the sampler alone is sufficient.

In [7]:
train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transforms)
val_dataset   = datasets.ImageFolder(f'{DATA_DIR}/val',   transform=val_test_transforms)
test_dataset  = datasets.ImageFolder(f'{DATA_DIR}/test',  transform=val_test_transforms)

targets = train_dataset.targets

class_counts = Counter(targets)

class_weights = {
    class_id: 1.0 / count
    for class_id, count in class_counts.items()
}

sample_weights = [
    class_weights[target]
    for target in targets
]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("Sampler created.")
print("Number of training samples:", len(sample_weights))
print("Number of classes:", len(class_counts))


Sampler created.
Number of training samples: 6887
Number of classes: 195


In [8]:
# Safety check: ImageFolder must use the same class-to-index mapping
# for train, validation, and test.
assert train_dataset.class_to_idx == val_dataset.class_to_idx, "Train and val class mapping mismatch!"
assert train_dataset.class_to_idx == test_dataset.class_to_idx, "Train and test class mapping mismatch!"

print("ImageFolder class mappings are consistent across train/val/test.")

# Save mapping from model output index to ImageFolder class name.
# Example: 42 -> "BMW_X5_SUV_2007"
idx_to_class = {
    idx: class_name
    for class_name, idx in train_dataset.class_to_idx.items()
}

# Save locally for this runtime
local_label_map_path = DATA_DIR / "imagefolder_label_map.json"

with open(local_label_map_path, "w") as f:
    json.dump(idx_to_class, f, indent=4)

# Save permanently to Google Drive
drive_label_map_path = PROJECT_DIR / "data" / "imagefolder_label_map.json"

with open(drive_label_map_path, "w") as f:
    json.dump(idx_to_class, f, indent=4)

print("Saved local ImageFolder label map to:", local_label_map_path)
print("Saved Drive ImageFolder label map to:", drive_label_map_path)

ImageFolder class mappings are consistent across train/val/test.
Saved local ImageFolder label map to: /content/twincar_data/imagefolder_label_map.json
Saved Drive ImageFolder label map to: /content/drive/MyDrive/twincar/data/imagefolder_label_map.json


In [9]:
NUM_WORKERS = 2 if os.cpu_count() > 1 else 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')
print(f'Classes       : {len(train_dataset.classes)}')

Train batches : 216
Val batches   : 38
Test batches  : 250
Classes       : 195


## Model: EfficientNet-B0

We use EfficientNet-B0 pretrained on ImageNet as our backbone for three reasons:

1. **Parameter efficiency**: B0 achieves 77.1% top-1 on ImageNet with 5.3M
   parameters vs ResNet-50's 26M at 76.0% — more accuracy per parameter.
2. **Small dataset fit**: Research on backbone selection for small-data fine-tuning
   consistently shows EfficientNet and ConvNeXt outperforming ViTs and ResNets
   in low-data regimes. ViTs in particular underperform when fine-tuned on
   limited data due to lacking CNN's spatial inductive biases.
3. **Colab constraints**: Lower memory footprint means larger batches fit in the
   T4 GPU's 15GB VRAM, which is important given we can't use multi-GPU.

Fine-tuning strategy: freeze backbone for epochs 1–5 (head warmup), then unfreeze
full network with a differential learning rate (backbone LR = head LR / 10).

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [11]:
NUM_CLASSES = len(train_dataset.classes)  # 195

model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

# Replace classifier head
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(in_features, NUM_CLASSES)
)

# Phase 1: freeze backbone, train head only
for param in model.features.parameters():
    param.requires_grad = False

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 146MB/s]


Trainable params: 249,795 / 4,257,343


In [12]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Phase 2 optimizer and scheduler will be initialized in 04_training.ipynb
# at the start of Phase 2, after backbone is unfrozen.

EPOCHS_PHASE1 = 5
EPOCHS_PHASE2 = 15
TOTAL_EPOCHS  = EPOCHS_PHASE1 + EPOCHS_PHASE2

# Phase 1 optimizer — head only
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-4
)

# Phase 1 scheduler — cosine over phase 1 only
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS_PHASE1, eta_min=1e-6
)

In [14]:
model.eval()

with torch.no_grad():
    batch_imgs, batch_labels = next(iter(train_loader))
    batch_imgs = batch_imgs.to(device)

    out = model(batch_imgs)

    print(f"Input shape  : {batch_imgs.shape}")
    print(f"Output shape : {out.shape}")
    print(f"Expected     : ({batch_imgs.size(0)}, {NUM_CLASSES})")

    assert out.shape == (batch_imgs.size(0), NUM_CLASSES), "Shape mismatch!"

    print("Forward pass OK")

model.train()

Input shape  : torch.Size([32, 3, 224, 224])
Output shape : torch.Size([32, 195])
Expected     : (32, 195)
Forward pass OK


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat